In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier
import torch
import torch.nn as nn
import pickle
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import lightgbm as lgb
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

In [ ]:
class RiskClassifier:
    def __init__(self):
        self.model = lgb.LGBMClassifier(random_state=42, is_unbalance=True)
        self.features = [
            'open', 'high', 'low', 'close', 'volume',
            'ema_50', 'ema_200', 'macd', 'macd_signal', 'macd_diff', 'rsi',
            'bollinger_hband', 'bollinger_lband', 'mfi',
            'return', 'bollinger_pct', 'bollinger_bw',
            'volatility_5d', 'volatility_10d',
            'volume_change', 'obv', 'volume_pct_20d', 'close_lag1', 'close_lag2',
            'close_lag3', 'return_lag1', 'return_lag2', 'return_lag3', 'rsi_lag1',
            'rsi_lag2', 'rsi_lag3', 'macd_lag1', 'macd_lag2', 'macd_lag3',
            'macd_diff_lag1', 'macd_diff_lag2', 'macd_diff_lag3',
            'bollinger_pct_lag1', 'bollinger_pct_lag2', 'bollinger_pct_lag3'
        ]

    def train(self, df_train_featured):
        df_train = df_train_featured.copy()
        
        test_size = 0.10
        all_Xtrain_dfs = []
        all_ytrain_dfs = []
        

        for ticker in df_train["ticker"].unique():
            df_ticker = df_train[df_train["ticker"] == ticker].copy()
            df_ticker.sort_values(by = "timestamp", inplace = True)
            split_index = int((len(df_ticker) * (1-test_size)))
            df_ticker_train = df_ticker[:split_index]
            all_Xtrain_dfs.append(df_ticker_train[self.features])
            all_ytrain_dfs.append(df_ticker_train["risk_label"])
        

        X_train = pd.concat(all_Xtrain_dfs)
        y_train = pd.concat(all_ytrain_dfs)
         
        self.model.fit(X_train, y_train)
        print("[Risk Model] Huấn luyện xong")

    def predict(self, df_today_featured):
        if df_today_featured.empty:
            return pd.Series(dtype=int)

        
        X_today = df_today_featured[self.features].dropna()
        if X_today.empty:
            return pd.Series(dtype=int)
            
        predictions = self.model.predict(X_today)
        return pd.Series(predictions, index=X_today.index)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_layer_size, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_layer_size = hidden_layer_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_layer_size, num_layers, 
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.linear = nn.Linear(hidden_layer_size * 2, num_classes)

    def forward(self, input_seq):
        device = input_seq.device
        h0 = torch.zeros(self.num_layers * 2, input_seq.size(0), self.hidden_layer_size).to(device)
        c0 = torch.zeros(self.num_layers * 2, input_seq.size(0), self.hidden_layer_size).to(device)
        lstm_out, _ = self.lstm(input_seq, (h0, c0))
        predictions = self.linear(lstm_out[:, -1, :])
        return predictions

In [ ]:
class SignalClassifier:
    def __init__(self, model_path, top_features, scaler_path,
                 time_steps=10, device="cpu"):
        self.model_path = model_path
        self.top_features = top_features
        self.scaler_path = scaler_path
        self.time_steps = time_steps
        self.device = device
        self.scaler = MinMaxScaler(feature_range=(0, 1))
        self.model = None

    def load_model(self, input_size, num_classes=3, hidden_size=128):
        with open(self.scaler_path, "rb") as f:
            self.scaler = pickle.load(f)

        model = LSTMClassifier(input_size=input_size, hidden_layer_size=hidden_size, num_classes=num_classes)
        model.load_state_dict(torch.load(self.model_path, map_location=self.device))
        model.to(self.device)
        model.eval()
        self.model = model
        print(f"[Signal Model] Load weight từ {self.model_path} thành công")
        print(f"[Signal Model] Load scaler từ {self.scaler_path} thành công")

    def _create_lstm_dataset(self, X_scaled):
        if len(X_scaled) < self.time_steps:
            return np.array([])
        return X_scaled[-self.time_steps:].reshape(1, self.time_steps, X_scaled.shape[1])

    def predict(self, df_history_featured):
        if df_history_featured.empty or len(df_history_featured) < self.time_steps:
            return 1 

        X_hist = df_history_featured[self.top_features].values
        X_hist_scaled = self.scaler.transform(X_hist)

        X_lstm = self._create_lstm_dataset(X_hist_scaled)
        if X_lstm.shape[0] == 0:
            return 1 
        
        X_tensor = torch.tensor(X_lstm, dtype=torch.float32).to(self.device)

        with torch.no_grad():
            output = self.model(X_tensor)
            pred = torch.argmax(output, dim=1).item()
        
        return pred

In [ ]:
def add_features(df):
    df_copy = df.copy()

    # Features cho Risk Model
    df_copy["volatility_5d"] = df_copy.groupby("ticker")["return"].rolling(5).std().reset_index(0, drop=True)
    df_copy["volatility_10d"] = df_copy.groupby("ticker")["return"].rolling(10).std().reset_index(0, drop=True)

    df_copy["volume_change"] = df_copy.groupby("ticker")["volume"].pct_change()
    
    df_copy["obv"] = (np.sign(df_copy["return"].fillna(0)) * df_copy["volume"]).groupby(df_copy["ticker"]).cumsum()
    
    df_copy["volume_pct_20d"] = df_copy.groupby("ticker")["volume"].transform(
        lambda x: x.rolling(20).apply(lambda s: pd.Series(s).rank(pct=True).iloc[-1])
    )

    lag_cols = ["close", "return", "rsi", "macd", "macd_diff", "bollinger_pct"]
    for col in lag_cols:
        for lag in [1, 2, 3]:
            df_copy[f"{col}_lag{lag}"] = df_copy.groupby("ticker")[col].shift(lag)
    
    # Features cho Signal Model
    lags = [5, 10, 15]
    for lag in lags:
        df_copy[f'return_lag_{lag}'] = df_copy.groupby('ticker')['return'].shift(lag)

    vol_windows = [10, 20]
    for window in vol_windows:
        df_copy[f'volatility_{window}'] = df_copy.groupby('ticker')['return'].rolling(window=window).std().reset_index(0, drop=True)

    df_copy['rsi_momentum'] = df_copy.groupby('ticker')['rsi'].diff()
    df_copy['macd_diff_momentum'] = df_copy.groupby('ticker')['macd_diff'].diff()
    df_copy['price_vs_ema200'] = (df_copy['close'] / df_copy['ema_200']) - 1

    indicators_to_lag = ['rsi', 'mfi']
    for indicator in indicators_to_lag:
        for lag in [1, 3]:
            df_copy[f'{indicator}_lag_{lag}'] = df_copy.groupby('ticker')[indicator].shift(lag)

    return df_copy

def calculate_risk_label(df):
    df['min_price_in_future'] = df.groupby('ticker')['low'].shift(-10).rolling(window=10).min()

    df['future_max_drawdown'] = (df['min_price_in_future'] / df["close"]) - 1

    df.dropna(subset=['future_max_drawdown'], inplace=True)

    df['T2_drawdown'] = df.groupby('timestamp')['future_max_drawdown'].transform(lambda x: x.quantile(0.35))

    labels = [1, 0]
    conditions = [
        df['future_max_drawdown'] <= df['T2_drawdown'],
        df['future_max_drawdown'] > df['T2_drawdown']
    ]
    df['risk_label'] = np.select(conditions, labels)

    return df


In [ ]:
def create_all_features(df, window_size=252):
    df_copy = df.copy()

    df_vnx = df[df["exchange"] == "HOSE"]
    df_hnx = df[df["exchange"] == "HNX"]
    df_upcom = df[df["exchange"] == "UPCOM"]

    df_vnx = calculate_risk_label(df_vnx)
    df_hnx = calculate_risk_label(df_hnx)
    df_upcom = calculate_risk_label(df_upcom)

    df_vnx = add_features(df_vnx)
    df_hnx = add_features(df_hnx)
    df_upcom = add_features(df_upcom)
    
    return df_vnx, df_hnx, df_upcom

In [ ]:
def filter_ticker(vnx_finance_path, hnx_finance_path, upcom_finance_path):
    list_qualified_tickers = []
    df_vnx = pd.read_excel(vnx_finance_path)
    df_hnx = pd.read_excel(hnx_finance_path)
    df_upcom = pd.read_excel(upcom_finance_path)

    for ticker in df_hnx["ticker"].unique():
        row = df_hnx[df_hnx["ticker"] == ticker].iloc[0]  
        if (
            row["EBITMargin"] >= 0.03
            and row["ROA"] >= 0.01
            and row["ROE"] >= 0.01
            and row["ROIC"] >= 0.02
        ):
            list_qualified_tickers.append(ticker)

    
    for ticker in df_vnx["ticker"].unique():
        row = df_vnx[df_vnx["ticker"] == ticker].iloc[0]  
        if (
            row["EBITMargin"] >= 0.05
            and row["ROA"] >= df_vnx["ROA"].quantile(0.25)
            and row["ROE"] >= df_vnx["ROE"].quantile(0.25)
            and row["ROIC"] >= 0.04
        ):
            list_qualified_tickers.append(ticker)

    for ticker in df_upcom["ticker"].unique():
        row = df_upcom[df_upcom["ticker"] == ticker].iloc[0]  
        if (
            row["EBITMargin"] >= df_upcom["EBITMargin"].quantile(0.35)
            and row["ROA"] >= df_upcom["ROA"].quantile(0.25)
            and row["ROE"] >= df_upcom["ROE"].quantile(0.25)
            and row["ROIC"] >= df_upcom["ROIC"].quantile(0.35)
        ):
            list_qualified_tickers.append(ticker)

    return list_qualified_tickers

VNX_FINANCE_PATH = r"/kaggle/input/vnx-finance/VNINDEX_Finance.xlsx"
HNX_FINANCE_PATH = r"/kaggle/input/hnx-finance/HNXINDEX_Finance.xlsx"
UPCOM_FINANCE_PATH = r"/kaggle/input/upcom-finance/UPCOMINDEX_Finance.xlsx"
list_qualified_tickers = filter_ticker(VNX_FINANCE_PATH, HNX_FINANCE_PATH, UPCOM_FINANCE_PATH)
print(len(list_qualified_tickers))

In [ ]:
def filter_ticker_by_indicators(data):
    # Ngưỡng chỉ số cho VN-Index
    vn_rsi_low, vn_rsi_high = 30, 70
    vn_macd_low, vn_macd_high = -190.806461, 191.497498
    vn_boll_low, vn_boll_high = 0.247301, 0.769456

    # Ngưỡng chỉ số cho HNX-Index 
    hnx_rsi_low, hnx_rsi_high = 25, 75
    hnx_macd_low, hnx_macd_high = -171.424148, 176.457420
    hnx_boll_low, hnx_boll_high = 0.230735, 0.735965

    # Ngưỡng chỉ số cho UPCOM 
    upcom_rsi_low, upcom_rsi_high = 20, 80
    upcom_macd_low, upcom_macd_high = -161.357600, 163.167889
    upcom_boll_low, upcom_boll_high = 0.224304, 0.733178

    # Lọc cổ phiếu cho từng sàn
    vn_filtered = data[
        (data['exchange'] == 'VN') &
        (data['rsi'] >= vn_rsi_low) & (data['rsi'] <= vn_rsi_high) &
        (data['macd_diff'] >= vn_macd_low) & (data['macd_diff'] <= vn_macd_high) &
        (data['bollinger_pct'] >= vn_boll_low) & (data['bollinger_pct'] <= vn_boll_high)
    ]

    hnx_filtered = data[
        (data['exchange'] == 'HNX') &
        (data['rsi'] >= hnx_rsi_low) & (data['rsi'] <= hnx_rsi_high) &
        (data['macd_diff'] >= hnx_macd_low) & (data['macd_diff'] <= hnx_macd_high) &
        (data['bollinger_pct'] >= hnx_boll_low) & (data['bollinger_pct'] <= hnx_boll_high)
    ]

    upcom_filtered = data[
        (data['exchange'] == 'UPCOM') &
        (data['rsi'] >= upcom_rsi_low) & (data['rsi'] <= upcom_rsi_high) &
        (data['macd_diff'] >= upcom_macd_low) & (data['macd_diff'] <= upcom_macd_high) &
        (data['bollinger_pct'] >= upcom_boll_low) & (data['bollinger_pct'] <= upcom_boll_high)
    ]


    filtered_df = pd.concat([vn_filtered, hnx_filtered, upcom_filtered], ignore_index=True)

    return filtered_df




In [ ]:
def run_backtest(all_df, RiskClassifier, SignalClassifier, vnindex_df, 
                 risk_model_params={},
                 signal_model_params={},
                 initial_balance=1_000_000_000,
                 trailing_stop_percent=0.07,
                 initial_stop_loss_percent=0.08,
                 min_hold_period=10,
                 max_hold_period=20,
                 full_take_profit_percent=0.20,
                 partial_take_profit_percent = 0.10,
                 partial_take_profit_amount = 0.5,
                 max_concurrent_positions=15,    
                ):
    # --- B0: Feature Engineering ---
    df_vnx, df_hnx, df_upcom = create_all_features(all_df)
    all_df_featured = pd.concat([df_vnx, df_hnx, df_upcom], ignore_index= True)
    all_df_featured.dropna(inplace=True)
    all_df_featured = all_df_featured.sort_values("timestamp").set_index("timestamp")

    # --- Chia train/test ---
    unique_dates = all_df_featured.index.unique().sort_values()
    split_idx = int(0.90 * len(unique_dates))
    train_cutoff = unique_dates[split_idx]
    train_df = all_df_featured.loc[all_df_featured.index <= train_cutoff]
    test_df_raw = all_df_featured.loc[all_df_featured.index > train_cutoff]

    dates_to_drop = [
        '2025-05-08', '2025-06-05', '2025-06-12', '2025-06-17',
        '2025-06-25', '2025-06-30', '2025-07-03'
    ]
    
   
    test_df_dates_str = pd.to_datetime(test_df_raw.index).strftime('%Y-%m-%d')
    
    test_df = test_df_raw[~np.isin(test_df_dates_str, dates_to_drop)]

    # --- Train Risk model ---
    risk_model_vnx = RiskClassifier(**risk_model_params)
    risk_model_vnx.train(df_vnx)

    risk_model_hnx = RiskClassifier(**risk_model_params)
    risk_model_hnx.train(df_hnx)

    risk_model_upcom = RiskClassifier(**risk_model_params)
    risk_model_upcom.train(df_upcom)

    # --- Load Signal model ---
    signal_model = SignalClassifier(**signal_model_params)
    input_size = len(signal_model.top_features)
    signal_model.load_model(input_size=input_size)

    all_trading_days = test_df.index.unique().sort_values()    
    balance = initial_balance
    trading_days = all_trading_days[:-3] 
    final_liquidation_date = all_trading_days[-3] 
    holdings = {}
    portfolio_history = []
    trade_log = []

    mdd_snapshots = [] # List để lưu tất cả các "ảnh chụp"
    was_in_deep_drawdown = False # Biến trạng thái để theo dõi

    portfolio_peak_value = initial_balance 

    
    
    print("\nBẮT ĐẦU BACKTEST")
    for date in tqdm(trading_days, desc="Backtesting"):    
        df_today_full_raw = test_df.loc[[date]] 
        df_today_full = df_today_full_raw.set_index('ticker')
        df_today_filtered_raw = df_today_full_raw.copy() # Bắt đầu từ bản đầy đủ
        if list_qualified_tickers:
            df_today_filtered_raw = df_today_filtered_raw[df_today_filtered_raw["ticker"].isin(list_qualified_tickers)]
        df_today_filtered_raw = filter_ticker_by_indicators(df_today_filtered_raw)
        df_today = df_today_filtered_raw.set_index('ticker')
        
       

        is_market_bullish = True 
        if date in vnindex_df.index and not np.isnan(vnindex_df.loc[date, 'sma200']):
            is_market_bullish = vnindex_df.loc[date, 'close'] > vnindex_df.loc[date, 'sma200']

        # --- 1. Risk & Signal Classification ---
        risk_preds = []

        mask_hose = df_today["exchange"] == "HOSE"
        if mask_hose.any():
            preds_hose = risk_model_vnx.predict(df_today.loc[mask_hose])
            risk_preds.extend(list(zip(df_today.loc[mask_hose].index, preds_hose)))

        mask_hnx = df_today["exchange"] == "HNX"
        if mask_hnx.any():
            preds_hnx = risk_model_hnx.predict(df_today.loc[mask_hnx])
            risk_preds.extend(list(zip(df_today.loc[mask_hnx].index, preds_hnx)))

        mask_upcom = df_today["exchange"] == "UPCOM"
        if mask_upcom.any():
            preds_upcom = risk_model_upcom.predict(df_today.loc[mask_upcom])
            risk_preds.extend(list(zip(df_today.loc[mask_upcom].index, preds_upcom)))

        risk_preds_series = pd.Series(dict(risk_preds))

        eligible_tickers = risk_preds_series[risk_preds_series == 0].index.tolist()
        if eligible_tickers:
            print(f"\n[{date}] Các mã đủ điều kiện: {len(eligible_tickers)} - {eligible_tickers}")
        
        signals = {}
        if eligible_tickers:
            for ticker in eligible_tickers:
                hist_data = all_df_featured[all_df_featured["ticker"] == ticker].loc[:date].copy()
                if not hist_data.empty:
                    signal = signal_model.predict(hist_data)
                    signals[ticker] = signal
        
        buy_signals = {t: s for t, s in signals.items() if s == 2}
        sell_signals = {t: s for t, s in signals.items() if s == 0}
        
        if eligible_tickers:
             print(f"\n[{date}] Mã đủ điều kiện rủi ro: {len(eligible_tickers)}")
        if signals:
            print(f"  -> Tín hiệu: Mua: {len(buy_signals)}, Bán: {len(sell_signals)}, Giữ: {len(signals) - len(buy_signals) - len(sell_signals)}")

        # --- 2. Quản lý Holdings ---
        current_holdings = list(holdings.keys())
        for ticker in current_holdings:
            if ticker in df_today_full.index:
                current_price = df_today_full.loc[ticker, "close"]
                entry_price = holdings[ticker]['entry_price']
                
                holdings[ticker]['high_since_entry'] = max(holdings[ticker]['high_since_entry'], current_price)
                holdings[ticker]['trailing_stop'] = holdings[ticker]['high_since_entry'] * (1 - trailing_stop_percent)
                holdings[ticker]['days_held'] += 1

                should_sell = False
                reason_to_sell = ""

                current_profit_percent = (current_price / entry_price) - 1
                if not holdings[ticker]['partial_profit_taken']:
                    if current_profit_percent >= partial_take_profit_percent:
                        shares_to_sell = int(holdings[ticker]['shares'] * partial_take_profit_amount) 
                        if shares_to_sell > 0:
                            balance += shares_to_sell * current_price
                            holdings[ticker]['shares'] -= shares_to_sell 
                            holdings[ticker]['partial_profit_taken'] = True
                            
                            trade_log.append({
                                'ticker': ticker, 'entry_date': holdings[ticker]['entry_date'], 'exit_date': date,
                                'entry_price': entry_price, 'exit_price': current_price, 'shares': shares_to_sell,
                                'trade_type': 'Partial Take Profit (Profit Target)'
                            })
                            print(f"  -> CHỐT LỜI 1/2: {shares_to_sell} {ticker} tại giá {current_price:,.0f} (Lãi {current_profit_percent:.2%})")
                
                if current_profit_percent >= full_take_profit_percent:
                    should_sell = True
                    reason_to_sell = f"Take Profit ({full_take_profit_percent:.0%})"
                

                elif current_price <= holdings[ticker].get('initial_stop_price', 0):
                    should_sell = True
                    reason_to_sell = "Initial Stop-loss"
                    
                elif current_price <= holdings[ticker]['trailing_stop']:
                    should_sell = True
                    reason_to_sell = "Trailing Stop"
                    
                    
                elif holdings[ticker]['days_held'] >= min_hold_period:
                    if signals.get(ticker, 1) == 0:
                        should_sell = True
                        reason_to_sell = "SignalClassifier BÁN"
                    elif holdings[ticker]['days_held'] >= max_hold_period:
                        should_sell = True
                        reason_to_sell = f"Max Hold Period ({max_hold_period} days)"
                        
                if should_sell and holdings[ticker]['shares'] > 0:
                    balance += holdings[ticker]['shares'] * current_price
                    trade_log.append({
                        'ticker': ticker, 'entry_date': holdings[ticker]['entry_date'], 'exit_date': date,
                        'entry_price': entry_price, 'exit_price': current_price, 'shares': holdings[ticker]['shares'],
                        'trade_type': reason_to_sell
                    })
                    print(f"  -> BÁN TOÀN BỘ: {holdings[ticker]['shares']} {ticker} tại giá {current_price:,.0f} ({reason_to_sell})")
                    del holdings[ticker]

        # Các bước mua

        buy_tickers_today = [t for t, s in buy_signals.items() if t not in holdings]

        if buy_tickers_today and is_market_bullish:
            slots_available = max_concurrent_positions - len(holdings)
            
            if slots_available <= 0:
                print("  -> Đã đạt giới hạn số lượng vị thế. Không mua mới.")
            else:
                buy_tickers_to_consider = buy_tickers_today[:slots_available]
                
                risk_per_trade_percent = 0.01 
                initial_stop_loss_percent = 0.08
                max_capital_deployment_per_day = 0.40
                
                # Tính tổng giá trị tài sản hiện tại (tiền mặt + giá trị cổ phiếu)
                holdings_value = 0
                for ticker_held, info_held in holdings.items():
                    if ticker_held in df_today_full.index:
                        holdings_value += info_held['shares'] * df_today_full.loc[ticker_held, "close"]
                total_portfolio_value = balance + holdings_value
                
                # Tính toán tổng vốn có thể triển khai trong ngày hôm nay
                capital_to_deploy_today = total_portfolio_value * max_capital_deployment_per_day
                
                print(f"  -> Ngân sách triển khai vốn ngày hôm nay: {capital_to_deploy_today:,.0f} VNĐ")
                
                for ticker in buy_tickers_to_consider: 
                    if capital_to_deploy_today <= 0:
                        print("  -> Đã hết ngân sách triển khai vốn cho ngày hôm nay.")
                        break 
        
                    if ticker in df_today_full.index:
                        price = df_today_full.loc[ticker, "close"]
                        
                        stop_loss_price = price * (1 - initial_stop_loss_percent)
                        
                        risk_per_share = price - stop_loss_price
        
                        if price > 0 and risk_per_share > 0:
                            capital_at_risk = total_portfolio_value * risk_per_trade_percent
                            
                            shares = int(capital_at_risk // risk_per_share)
                            
                            position_value = shares * price
                            
                            if shares > 0 and position_value <= balance and position_value <= capital_to_deploy_today:
                                balance -= position_value
                                capital_to_deploy_today -= position_value # Trừ vào ngân sách ngày
                                
                                holdings[ticker] = {
                                    'shares': shares, 
                                    'entry_price': price, 
                                    'entry_date': date, 
                                    'high_since_entry': price,
                                    'trailing_stop': price * (1 - trailing_stop_percent),
                                    'initial_stop_price': stop_loss_price,
                                    'days_held': 0, 
                                    'partial_profit_taken': False, 
                                    'status': 'initial'
                                }
                                print(f"  -> MUA: {shares} {ticker} tại giá {price:,.0f} (Vốn: {position_value:,.0f} VNĐ, Rủi ro: {capital_at_risk:,.0f} VNĐ)")
                            
                            elif position_value > balance:
                                print(f"  [CẢNH BÁO] Không đủ tiền mặt để mua {ticker}. Cần {position_value:,.0f}, còn {balance:,.0f}")
                            elif position_value > capital_to_deploy_today:
                                print(f"  [THÔNG TIN] Bỏ qua mua {ticker} vì vượt quá ngân sách triển khai vốn còn lại trong ngày.")
        

        elif not is_market_bullish:
            print(f"  -> [{date.date()}] Bỏ qua tín hiệu mua do thị trường chung xấu.")


        holdings_value = 0
        for ticker, info in holdings.items():
            if ticker in df_today_full.index:
                holdings_value += info['shares'] * df_today_full.loc[ticker, "close"]
        total_value = balance + holdings_value
        portfolio_history.append({'date': date, 'value': total_value})
        
        portfolio_peak_value = max(portfolio_peak_value, total_value)
        
        # Tính toán mức sụt giảm hiện tại từ đỉnh
        current_drawdown = (total_value - portfolio_peak_value) / portfolio_peak_value

        mdd_threshold = -0.50 # Ngưỡng 50%

        # Điều kiện: Vừa mới sụt giảm xuống dưới ngưỡng
        if current_drawdown <= mdd_threshold and not was_in_deep_drawdown:
            print(f"\n{'#'*10} CẢNH BÁO: DANH MỤC VỪA SỤT GIẢM VƯỢT {mdd_threshold:.0%} {'#'*10}")
            
            # Tạo một "ảnh chụp" mới
            current_snapshot = {
                'date': date,
                'drawdown': current_drawdown,
                'holdings_at_mdd': []
            }
            
            # Lặp qua các cổ phiếu đang nắm giữ tại thời điểm đó
            for ticker_held, info_held in holdings.items():
                current_price_at_mdd = df_today_full.loc[ticker_held, "close"] if ticker_held in df_today_full.index else np.nan
                current_snapshot['holdings_at_mdd'].append({
                    'ticker': ticker_held,
                    'entry_price': info_held['entry_price'],
                    'current_price': current_price_at_mdd
                })

            # Thêm ảnh chụp vào danh sách và cập nhật trạng thái
            mdd_snapshots.append(current_snapshot)
            was_in_deep_drawdown = True
        
        # Điều kiện: Vừa mới phục hồi lên trên ngưỡng
        elif current_drawdown > mdd_threshold and was_in_deep_drawdown:
            # Reset trạng thái để sẵn sàng ghi nhận cho lần sụt giảm tiếp theo
            was_in_deep_drawdown = False

    df_final_day_raw = test_df.loc[final_liquidation_date]

    if isinstance(df_final_day_raw, pd.Series):
        df_final_day = df_final_day_raw.to_frame().T
    else:
        df_final_day = df_final_day_raw.copy()
    df_final_day = df_final_day.set_index('ticker')

    for ticker, info in list(holdings.items()):
        if ticker in df_final_day.index and not np.isnan(df_final_day.loc[ticker, "close"]):
            final_price = df_final_day.loc[ticker, "close"]
            balance += info['shares'] * final_price
            trade_log.append({
                'ticker': ticker, 'entry_date': info['entry_date'], 'exit_date': final_liquidation_date,
                'entry_price': info['entry_price'], 'exit_price': final_price, 'shares': info['shares'],
                'trade_type': 'Final Liquidation'
            })
        else:
            print(f"[CẢNH BÁO] Mã {ticker} không có dữ liệu vào ngày thanh lý {final_liquidation_date}. Ghi nhận lỗ.")
            trade_log.append({
                'ticker': ticker, 'entry_date': info['entry_date'], 'exit_date': final_liquidation_date,
                'entry_price': info['entry_price'], 'exit_price': 0, 'shares': info['shares'],
                'trade_type': 'Liquidation Failed (Assumed 0)'
            })

    final_total_value = balance

    # Tạo một dòng cuối cùng trong lịch sử danh mục cho ngày thanh lý
    if portfolio_history:
    # Lấy ngày giao dịch cuối cùng (ngày áp chót)
        last_trading_day_entry = portfolio_history[-1]
        # Cập nhật giá trị của ngày đó bằng giá trị sau khi đã thanh lý
        last_trading_day_entry['value'] = final_total_value
        # Ghi đè lại vào danh sách
        portfolio_history[-1] = last_trading_day_entry

    if mdd_snapshots:
        print("\n" + "="*80)
        print("--- PHÂN TÍCH CÁC THỜI ĐIỂM DANH MỤC SỤT GIẢM > 50% ---")
        print("="*80)
        
        # Lặp qua từng lần sụt giảm đã ghi nhận
        for i, snapshot in enumerate(mdd_snapshots, 1):
            mdd_date = snapshot['date']
            mdd_percent = snapshot['drawdown']
            
            print(f"\nLẦN SỤT GIẢM THỨ {i}:")
            print(f"Ngày ghi nhận: {mdd_date} (Mức sụt giảm lúc đó: {mdd_percent:.2%})")
            print("-" * 70)
            print(f"{'Cổ phiếu':<10} {'Giá vào lệnh':>15} {'Giá tại thời điểm sụt giảm':>30} {'Lãi/Lỗ':>12}")
            print(f"{'-'*10} {'-'*15} {'-'*30} {'-'*12}")
            
            for stock_info in snapshot['holdings_at_mdd']:
                ticker = stock_info['ticker']
                entry_price = stock_info['entry_price']
                current_price = stock_info['current_price']
                pnl_percent = (current_price / entry_price - 1) * 100 if entry_price > 0 and not np.isnan(current_price) else 0
                
                print(f"{ticker:<10} {entry_price:,.0f}{' VND':>10} {current_price:,.0f}{' VND':>20} {pnl_percent:>10.2f}%")
            print("-" * 70)
    else:
        print("\n[THÔNG TIN] Mức sụt giảm tối đa của chiến lược không vượt quá 50%.")

# Chuyển đổi thành DataFrame SAU KHI đã cập nhật
    portfolio_history_df = pd.DataFrame(portfolio_history).set_index('date')

# Trả về kết quả
    return portfolio_history_df, pd.DataFrame(trade_log)

In [ ]:
import yfinance as yf
import numpy as np 

def analyze_performance(portfolio_df, trade_log_df, initial_balance):
    portfolio_df = portfolio_df.copy()
    trade_log_df = trade_log_df.copy() if not trade_log_df.empty else trade_log_df

    if portfolio_df.empty:
        print("Không có dữ liệu hiệu suất để phân tích.")
        return

    print("\n--- PHÂN TÍCH HIỆU SUẤT CHIẾN LƯỢC ---")
    
    # 1. Các chỉ số cơ bản
    final_value = portfolio_df['value'].iloc[-1]
    total_return_pct = (final_value / initial_balance - 1) * 100
    
    print(f"Vốn ban đầu:              {initial_balance:,.0f} VND")
    print(f"Giá trị cuối cùng:           {final_value:,.0f} VND")
    print(f"Tổng lợi nhuận:            {total_return_pct:.2f}%")

    # 2. Phân tích giao dịch
    if not trade_log_df.empty:
        total_trades = len(trade_log_df)
        trade_log_df['pnl'] = (trade_log_df['exit_price'] - trade_log_df['entry_price']) * trade_log_df['shares']
        winning_trades = trade_log_df[trade_log_df['pnl'] > 0]
        win_rate = (len(winning_trades) / total_trades) * 100 if total_trades > 0 else 0.0
        
        print(f"Tổng số giao dịch:        {total_trades}")
        print(f"Tỷ lệ chiến thắng (Win Rate): {win_rate:.2f}%")
    else:
        print("Tổng số giao dịch:        0")
        print("Tỷ lệ chiến thắng (Win Rate): N/A")

    # 3. Mức sụt giảm tối đa (Maximum Drawdown)
    portfolio_df['peak'] = portfolio_df['value'].cummax()
    portfolio_df['drawdown'] = (portfolio_df['value'] - portfolio_df['peak']) / portfolio_df['peak']
    max_drawdown = portfolio_df['drawdown'].min() * 100
    
    print(f"Mức sụt giảm tối đa (Max DD): {max_drawdown:.2f}%")

    

In [ ]:
DATA_PATH_UPCOM = r"/kaggle/input/upcom-cleaned/UPCOM_cleaned.xlsx"
DATA_PATH_HOSE = r"/kaggle/input/vnx-cleaned/VNINDEX_cleaned.xlsx"
DATA_PATH_HNX = r"/kaggle/input/hnx-cleaned/HNXINDEX_cleaned.xlsx"
MODEL_PATH = r'/kaggle/input/lstm-weight/best_lstm_classifier_improved.pth'
SCALER_PATH = r"/kaggle/input/scaler-weight/feature_scaler.pkl"


data_upcom = pd.read_excel(DATA_PATH_UPCOM)
data_hose = pd.read_excel(DATA_PATH_HOSE)
data_hnx = pd.read_excel(DATA_PATH_HNX)
data_hose["exchange"] = "HOSE"
data_hnx["exchange"] = "HNX"
data_upcom["exchange"] = "UPCOM"
data_full = pd.concat([data_upcom, data_hose, data_hnx], ignore_index= True)



if data_full is not None:
    signal_model_params = {
        "top_features": ['price_vs_ema200', 'volatility_20', 'volume', 'macd_diff', 'bollinger_bw', 'ema_200', 'volatility_10', 'macd_signal', 'rsi', 'bollinger_pct', 'macd', 'ema_50', 'bollinger_lband', 'mfi_lag_3', 'bollinger_hband'],
        "time_steps": 10, "device": "cpu", "model_path": MODEL_PATH, "scaler_path": SCALER_PATH
    }


    result_df, trade_log_df = run_backtest(
        all_df=data_full,
        RiskClassifier=RiskClassifier,
        vnindex_df = data_hose,
        SignalClassifier=SignalClassifier,
        signal_model_params=signal_model_params,
        initial_balance=1_000_000_000 
    )

    if not result_df.empty:
        analyze_performance(
            portfolio_df=result_df,
            trade_log_df=trade_log_df,
            initial_balance=1_000_000_000 
        )
        
        print("\n--- BIỂU ĐỒ HIỆU SUẤT ---")
        result_df['value'].plot(figsize=(15, 7), title='Hiệu suất danh mục theo thời gian')
        plt.xlabel("Date")
        plt.ylabel("Portfolio Value (VND)")
        plt.grid(True)
        plt.show()